In [ ]:
%pip install -q vllm sentence-transformers tqdm implicit lightfm

In [ ]:
import os

os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'

import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# ===== Константы =====
DATA_DIR = Path('../../data/children_products')
CLEANED_CSV = DATA_DIR / 'clildren_product_cleaned.csv'
RAW_CSV = DATA_DIR / '!03&04_17_VSE.csv'

MIN_INTERACTIONS = 5
TRAIN_SPLIT = 0.8

N_EVAL_USERS = 1000
K_CANDIDATES = 50
FINAL_K = 20
MAX_HISTORY_ITEMS = 20

EMBED_MODEL = 'sentence-transformers/paraphrase-multilingual-mpnet-base-v2'
LLM_MODEL = 'NousResearch/Meta-Llama-3.1-8B-Instruct'

SKU_EMB_CACHE = DATA_DIR / 'sku_embeddings.npy'
LLM_RERANK_CACHE = DATA_DIR / 'llm_rerank_cache_vllm.json'
ZERO_SHOT_CACHE = DATA_DIR / 'llm_recommendations_cache_vllm.json'
CANDIDATES_CACHE = DATA_DIR / 'rerank_base_candidates.npz'
NCF_CHECKPOINT = DATA_DIR / 'ncf_state.pt'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f'Исходный датасет: {df.shape}')

raw = pd.read_csv(
    RAW_CSV,
    sep=';', encoding='cp1251',
    usecols=['ID_SKU', 'Номенклатура', 'Группа2', 'Группа3', 'Тип'],
    dtype=str, low_memory=False
)
raw = raw.dropna(subset=['ID_SKU']).drop_duplicates('ID_SKU')

df = df.merge(raw[['ID_SKU', 'Номенклатура']], on='ID_SKU', how='left')
fallback = df['Группа3'].fillna('') + ' / ' + df['Тип'].fillna('')
df['Номенклатура'] = df['Номенклатура'].fillna(fallback)

df_filtered = df[(df['Статус'] == 'Доставлен') & (df['Отменено'] == 'Нет')].copy()
df_filtered = df_filtered.dropna(subset=['Телефон_new', 'ID_SKU', 'Дата'])
df_filtered['Дата'] = pd.to_datetime(df_filtered['Дата'], errors='coerce')
df_filtered = df_filtered.dropna(subset=['Дата'])

for _ in range(5):
    uc = df_filtered.groupby('Телефон_new').size()
    ic = df_filtered.groupby('ID_SKU').size()
    active_users = uc[uc >= MIN_INTERACTIONS].index
    active_items = ic[ic >= MIN_INTERACTIONS].index
    before = len(df_filtered)
    df_filtered = df_filtered[
        df_filtered['Телефон_new'].isin(active_users) &
        df_filtered['ID_SKU'].isin(active_items)
    ]
    if len(df_filtered) == before:
        break

user_enc, item_enc = LabelEncoder(), LabelEncoder()
df_filtered['user_id'] = user_enc.fit_transform(df_filtered['Телефон_new'])
df_filtered['item_id'] = item_enc.fit_transform(df_filtered['ID_SKU'])
n_users, n_items = len(user_enc.classes_), len(item_enc.classes_)
print(f'Пользователей: {n_users:,}  Товаров: {n_items:,}  Взаимодействий: {len(df_filtered):,}')

df_filtered = df_filtered.sort_values('Дата')
split_date = df_filtered['Дата'].quantile(TRAIN_SPLIT)
train_data = df_filtered[df_filtered['Дата'] < split_date].copy()
test_data = df_filtered[df_filtered['Дата'] >= split_date].copy()
print(f'Split date: {split_date}')
print(f'Train: {len(train_data):,}  Test: {len(test_data):,}')

train_int = train_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
test_int = test_data.groupby(['user_id', 'item_id']).size().reset_index(name='count')
train_matrix = csr_matrix(
    (train_int['count'].values, (train_int['user_id'].values, train_int['item_id'].values)),
    shape=(n_users, n_items)
)

test_user_items = test_int.groupby('user_id')['item_id'].apply(list).to_dict()
train_user_set = set(train_int['user_id'].unique())
warm_eval_users = [u for u in test_user_items if u in train_user_set]

rng = np.random.default_rng(RANDOM_SEED)
sample_users = rng.choice(warm_eval_users, size=min(N_EVAL_USERS, len(warm_eval_users)), replace=False)
sample_users = sorted(map(int, sample_users))
print(f'Подвыборка для оценки: {len(sample_users)} пользователей')

In [ ]:
item_name_map = (
    df_filtered.groupby('item_id')['Номенклатура']
    .agg(lambda s: s.mode().iloc[0] if len(s.mode()) else s.iloc[0])
)
item_names = [item_name_map.loc[i] for i in range(n_items)]

if SKU_EMB_CACHE.exists():
    sku_emb = np.load(SKU_EMB_CACHE)
    if sku_emb.shape[0] != n_items:
        sku_emb = None
else:
    sku_emb = None

encoder = SentenceTransformer(EMBED_MODEL)
if sku_emb is None:
    sku_emb = encoder.encode(item_names, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    np.save(SKU_EMB_CACHE, sku_emb)

knn = NearestNeighbors(metric='cosine', algorithm='brute').fit(sku_emb)
print(f'SKU embeddings: {sku_emb.shape}')

In [ ]:
train_sorted = train_data.sort_values('Дата')
user_history_cache = {}
for uid, g in train_sorted.groupby('user_id'):
    user_history_cache[int(uid)] = list(zip(g['Дата'].dt.date.astype(str), g['Номенклатура']))
user_train_items_cache = {
    int(uid): set(g['item_id'].tolist())
    for uid, g in train_sorted.groupby('user_id')
}

def build_user_history(user_id: int, max_items: int = MAX_HISTORY_ITEMS) -> str:
    items = user_history_cache.get(user_id, [])
    tail = items[-max_items:]
    return '\n'.join(f'- {date}: {name}' for date, name in tail)

In [ ]:
try:
    from implicit.gpu.als import AlternatingLeastSquares
except ImportError:
    from implicit.als import AlternatingLeastSquares

als = AlternatingLeastSquares(
    factors=50, regularization=0.01, iterations=15,
    calculate_training_loss=True, random_state=RANDOM_SEED
)
als.fit(train_matrix)

als_candidates = {}
for uid in tqdm(sample_users, desc='ALS top-50'):
    ids, scores = als.recommend(
        uid, train_matrix[uid],
        N=K_CANDIDATES, filter_already_liked_items=True
    )
    als_candidates[int(uid)] = list(zip([int(x) for x in ids], [float(s) for s in scores]))

print(f'ALS candidates готовы для {len(als_candidates)} пользователей')
print(f'  пример: uid={sample_users[0]}, top-5 = {als_candidates[sample_users[0]][:5]}')

In [ ]:
df_for_neg = pd.read_csv(CLEANED_CSV)
df_hard = df_for_neg[
    df_for_neg['Статус'].isin(['Отменен', 'Возврат']) | (df_for_neg['Отменено'] == 'Да')
].copy()
df_hard = df_hard.dropna(subset=['Телефон_new', 'ID_SKU'])

phone_to_uid = dict(zip(df_filtered['Телефон_new'], df_filtered['user_id']))
sku_to_iid = dict(zip(df_filtered['ID_SKU'], df_filtered['item_id']))
df_hard['user_id'] = df_hard['Телефон_new'].map(phone_to_uid)
df_hard['item_id'] = df_hard['ID_SKU'].map(sku_to_iid)
df_hard = df_hard.dropna(subset=['user_id', 'item_id'])
df_hard['user_id'] = df_hard['user_id'].astype(int)
df_hard['item_id'] = df_hard['item_id'].astype(int)

positives = train_int[['user_id', 'item_id']].copy()
positives['label'] = 1.0

hard_negs = df_hard[['user_id', 'item_id']].drop_duplicates().copy()
hard_negs['label'] = 0.0

n_easy = len(positives) * 4
positive_pairs = set(zip(positives['user_id'].astype(int), positives['item_id'].astype(int)))
rng_neg = np.random.default_rng(RANDOM_SEED)
easy_users, easy_items = [], []
needed = n_easy
while len(easy_users) < n_easy:
    u_chunk = rng_neg.integers(0, n_users, size=needed * 2)
    i_chunk = rng_neg.integers(0, n_items, size=needed * 2)
    for u, i in zip(u_chunk.tolist(), i_chunk.tolist()):
        if (u, i) not in positive_pairs:
            easy_users.append(u)
            easy_items.append(i)
            if len(easy_users) >= n_easy:
                break
    needed = n_easy - len(easy_users)

easy_negs = pd.DataFrame({'user_id': easy_users, 'item_id': easy_items, 'label': 0.0})

train_samples = pd.concat([
    positives[['user_id','item_id','label']],
    hard_negs[['user_id','item_id','label']],
    easy_negs[['user_id','item_id','label']]
], ignore_index=True)
train_samples = train_samples.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'NCF train: positives={len(positives):,}, hard_neg={len(hard_negs):,}, '
      f'easy_neg={len(easy_negs):,}, total={len(train_samples):,}')

In [ ]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=32, hidden_dim=32, dropout=0.3):
        super().__init__()
        self.user_embedding_gmf = nn.Embedding(num_users, emb_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, emb_dim)

        self.user_embedding_mlp = nn.Embedding(num_users, emb_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, emb_dim)

        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, hidden_dim * 2),
            nn.BatchNorm1d(hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
        )
        self.fc_out = nn.Linear(emb_dim + hidden_dim // 2, 1)

    def forward(self, user_ids, item_ids):
        u_gmf = self.user_embedding_gmf(user_ids)
        v_gmf = self.item_embedding_gmf(item_ids)
        gmf_out = u_gmf * v_gmf

        u_mlp = self.user_embedding_mlp(user_ids)
        v_mlp = self.item_embedding_mlp(item_ids)
        mlp_out = self.mlp(torch.cat([u_mlp, v_mlp], dim=-1))

        x = torch.cat([gmf_out, mlp_out], dim=-1)
        logits = self.fc_out(x).squeeze(-1)
        return torch.sigmoid(logits)


class NCFDataset(Dataset):
    def __init__(self, user_ids, item_ids, labels):
        self.user_ids = torch.LongTensor(user_ids)
        self.item_ids = torch.LongTensor(item_ids)
        self.labels = torch.FloatTensor(labels)

    def __len__(self):
        return len(self.user_ids)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

ncf_model = NCF(n_users, n_items, emb_dim=32, hidden_dim=32, dropout=0.3).to(device)

if NCF_CHECKPOINT.exists():
    ncf_model.load_state_dict(torch.load(NCF_CHECKPOINT, map_location=device))
    print(f'NCF загружен из {NCF_CHECKPOINT}')
else:
    train_ds = NCFDataset(
        train_samples['user_id'].values.astype(np.int64),
        train_samples['item_id'].values.astype(np.int64),
        train_samples['label'].values.astype(np.float32),
    )
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, num_workers=0)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(ncf_model.parameters(), lr=5e-4, weight_decay=1e-5)
    n_epochs = 15
    for epoch in range(n_epochs):
        ncf_model.train()
        total_loss, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f'NCF ep {epoch+1}/{n_epochs}')
        for u, i, y in pbar:
            u, i, y = u.to(device), i.to(device), y.to(device)
            preds = ncf_model(u, i)
            loss = criterion(preds, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({'loss': f'{total_loss / n_batches:.4f}'})
    torch.save(ncf_model.state_dict(), NCF_CHECKPOINT)
    print(f'NCF сохранён в {NCF_CHECKPOINT}')

In [ ]:
ncf_model.eval()
ncf_candidates = {}
items_t = torch.arange(n_items, device=device, dtype=torch.long)

with torch.no_grad():
    for uid in tqdm(sample_users, desc='NCF top-50'):
        users_t = torch.full_like(items_t, fill_value=int(uid))
        scores = ncf_model(users_t, items_t).cpu().numpy()
        scores = np.nan_to_num(scores, nan=-np.inf)
        bought = user_train_items_cache.get(int(uid), set())
        if bought:
            scores[list(bought)] = -np.inf
        top_idx = np.argpartition(-scores, K_CANDIDATES)[:K_CANDIDATES]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        ncf_candidates[int(uid)] = [(int(i), float(scores[i])) for i in top_idx]

print(f'NCF candidates готовы для {len(ncf_candidates)} пользователей')
print(f'  пример: uid={sample_users[0]}, top-5 = {ncf_candidates[sample_users[0]][:5]}')

In [ ]:
from lightfm import LightFM

lfm = LightFM(
    no_components=64, loss='warp', learning_rate=0.05,
    item_alpha=1e-6, user_alpha=1e-6, random_state=RANDOM_SEED
)
n_lfm_epochs = 20
for ep in tqdm(range(n_lfm_epochs), desc='LightFM epochs'):
    lfm.fit_partial(train_matrix, epochs=1, num_threads=16)

lfm_candidates = {}
all_items = np.arange(n_items)
for uid in tqdm(sample_users, desc='LightFM top-50'):
    scores = lfm.predict(int(uid), all_items)
    scores = np.nan_to_num(scores, nan=-np.inf)
    bought = user_train_items_cache.get(int(uid), set())
    if bought:
        scores[list(bought)] = -np.inf
    top_idx = np.argpartition(-scores, K_CANDIDATES)[:K_CANDIDATES]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    lfm_candidates[int(uid)] = [(int(i), float(scores[i])) for i in top_idx]

print(f'LightFM candidates готовы для {len(lfm_candidates)} пользователей')
print(f'  пример: uid={sample_users[0]}, top-5 = {lfm_candidates[sample_users[0]][:5]}')

In [ ]:

def save_candidates(path, als_c, ncf_c, lfm_c, users):
    arr_users = np.array([int(u) for u in users], dtype=np.int64)
    np.savez(
        path,
        users=arr_users,
        als_ids=np.array([[i for i, _ in als_c[int(u)]] for u in users], dtype=np.int64),
        als_scores=np.array([[s for _, s in als_c[int(u)]] for u in users], dtype=np.float32),
        ncf_ids=np.array([[i for i, _ in ncf_c[int(u)]] for u in users], dtype=np.int64),
        ncf_scores=np.array([[s for _, s in ncf_c[int(u)]] for u in users], dtype=np.float32),
        lfm_ids=np.array([[i for i, _ in lfm_c[int(u)]] for u in users], dtype=np.int64),
        lfm_scores=np.array([[s for _, s in lfm_c[int(u)]] for u in users], dtype=np.float32),
    )

save_candidates(CANDIDATES_CACHE, als_candidates, ncf_candidates, lfm_candidates, sample_users)
print(f'Кандидаты сохранены в {CANDIDATES_CACHE}')

In [ ]:

try:
    del ncf_model
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

_eot_id = tokenizer.convert_tokens_to_ids('<|eot_id|>')
LLM_TERMINATORS = sorted({
    int(t) for t in [tokenizer.eos_token_id, _eot_id]
    if isinstance(t, int) and t is not None and t >= 0
})
print(f'Terminators: {LLM_TERMINATORS}')

llm = LLM(
    model=LLM_MODEL,
    dtype='bfloat16',
    max_model_len=8192,
    gpu_memory_utilization=0.9,
    enforce_eager=False,
    trust_remote_code=False,
)
print('vLLM loaded')

In [ ]:
SYSTEM_PROMPT_RERANK = (
    'Ты — ассистент-рекомендатель детского интернет-магазина. '
    'Тебе даны: история покупок пользователя и пронумерованный список из {n} товаров-кандидатов '
    'от рекомендательной системы. Твоя задача — переупорядочить кандидатов от самого вероятного '
    'к покупке к наименее вероятному, опираясь на возраст ребёнка, бренды, категории, ценовой сегмент. '
    'Используй ТОЛЬКО номера из списка кандидатов, не повторяй их и не пропускай ни один номер. '
    'Отвечай СТРОГО в формате JSON без дополнительного текста: '
    '{{"ranked": [номер1, номер2, ..., номер{n}]}}.'
)


def build_rerank_prompt(user_id: int, candidate_item_ids):
    hist = build_user_history(user_id, max_items=MAX_HISTORY_ITEMS)
    cand_lines = '\n'.join(
        f'{idx + 1}. {item_name_map.loc[iid]}'
        for idx, iid in enumerate(candidate_item_ids)
    )
    n = len(candidate_item_ids)
    user_prompt = (
        f'История покупок пользователя:\n{hist}\n\n'
        f'Кандидаты для переупорядочивания (всего {n}):\n{cand_lines}\n\n'
        f'Верни JSON с полем "ranked" — массивом из {n} разных номеров (1..{n}) '
        f'в порядке от самого подходящего к наименее. Только JSON, без поясняющего текста.'
    )
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT_RERANK.format(n=n)},
        {'role': 'user', 'content': user_prompt},
    ]
    return tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

In [ ]:
def parse_ranked_indices(raw_text: str, n_candidates: int):
    """Возвращает 0-индексированную перестановку длины n_candidates,
    или None при невосстановимой ошибке. Дозаполняет хвостом исходного порядка."""
    arr = None
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict):
            for key in ('ranked', 'order', 'indices', 'rank'):
                if key in obj and isinstance(obj[key], list):
                    arr = obj[key]
                    break
            if arr is None:
                for v in obj.values():
                    if isinstance(v, list):
                        arr = v
                        break
        elif isinstance(obj, list):
            arr = obj
    except Exception:
        m = re.search(r'\[[\s\S]*?\]', raw_text)
        if m:
            try:
                arr = json.loads(m.group(0))
            except Exception:
                arr = None

    if arr is None:
        return None

    seen = set()
    result = []
    for x in arr:
        try:
            i = int(x) - 1
        except Exception:
            continue
        if 0 <= i < n_candidates and i not in seen:
            seen.add(i)
            result.append(i)

    if not result:
        return None

    for i in range(n_candidates):
        if i not in seen:
            result.append(i)
    return result


def apply_rerank(candidate_ids, raw_text):
    """Применяет порядок от LLM. При ошибке — возвращает исходный порядок."""
    perm = parse_ranked_indices(raw_text, len(candidate_ids))
    if perm is None:
        return list(candidate_ids)
    return [candidate_ids[i] for i in perm]

In [ ]:
rerank_cache = {
    'als_raw': {}, 'ncf_raw': {}, 'lightfm_raw': {},
    'als_recs': {}, 'ncf_recs': {}, 'lightfm_recs': {},
}
if LLM_RERANK_CACHE.exists():
    with open(LLM_RERANK_CACHE, 'r', encoding='utf-8') as f:
        loaded = json.load(f)
    for k in rerank_cache:
        rerank_cache[k] = {int(uid): v for uid, v in loaded.get(k, {}).items()}
    n_raw = sum(len(v) for k, v in rerank_cache.items() if k.endswith('_raw'))
    print(f'Cache: восстановлено {n_raw} сырых ответов')

sources = [
    ('als',     als_candidates,  'als_raw'),
    ('ncf',     ncf_candidates,  'ncf_raw'),
    ('lightfm', lfm_candidates,  'lightfm_raw'),
]

tasks = []
prompts = []
for src_name, cand_dict, raw_key in sources:
    for uid in sample_users:
        uid = int(uid)
        if uid in rerank_cache[raw_key]:
            continue
        cand_ids = [iid for iid, _ in cand_dict[uid]]
        tasks.append((raw_key, uid, cand_ids))
        prompts.append(build_rerank_prompt(uid, cand_ids))

print(f'К обработке промптов: {len(prompts)}')

if prompts:
    sample_lens = [len(tokenizer.encode(p)) for p in prompts[:50]]
    p50, p95, p99 = np.percentile(sample_lens, [50, 95, 99])
    print(f'Длина промпта (по первым 50): p50={p50:.0f}, p95={p95:.0f}, p99={p99:.0f} токенов')

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=400,
        stop_token_ids=LLM_TERMINATORS,
    )

    t0 = time.time()
    outputs = llm.generate(prompts, sampling_params)
    elapsed = time.time() - t0
    total_gen = sum(len(o.outputs[0].token_ids) for o in outputs)
    print(f'\nГенерация {len(prompts)} промптов за {elapsed:.1f}s '
          f'({elapsed / len(prompts):.2f}s/промпт, {total_gen / elapsed:.0f} tok/sec total)')

    for (raw_key, uid, cand_ids), out in zip(tasks, outputs):
        rerank_cache[raw_key][uid] = out.outputs[0].text

with open(LLM_RERANK_CACHE, 'w', encoding='utf-8') as f:
    json.dump(
        {k: {str(uid): v for uid, v in d.items()} for k, d in rerank_cache.items()},
        f, ensure_ascii=False
    )
print(f'\nКэш сохранён в {LLM_RERANK_CACHE}')

In [ ]:
def materialize_recs(cand_dict, raw_dict):
    out = {}
    for uid in sample_users:
        uid = int(uid)
        cand_ids = [iid for iid, _ in cand_dict[uid]]
        raw_text = raw_dict.get(uid, '')
        reranked = apply_rerank(cand_ids, raw_text)
        out[uid] = reranked[:FINAL_K]
    return out

als_recs_rerank = materialize_recs(als_candidates, rerank_cache['als_raw'])
ncf_recs_rerank = materialize_recs(ncf_candidates, rerank_cache['ncf_raw'])
lfm_recs_rerank = materialize_recs(lfm_candidates, rerank_cache['lightfm_raw'])

rerank_cache['als_recs']     = als_recs_rerank
rerank_cache['ncf_recs']     = ncf_recs_rerank
rerank_cache['lightfm_recs'] = lfm_recs_rerank
with open(LLM_RERANK_CACHE, 'w', encoding='utf-8') as f:
    json.dump(
        {k: {str(uid): v for uid, v in d.items()} for k, d in rerank_cache.items()},
        f, ensure_ascii=False
    )

def fallback_count(raw_dict):
    n_total, n_fb = 0, 0
    for uid in sample_users:
        n_total += 1
        if parse_ranked_indices(raw_dict.get(int(uid), ''), K_CANDIDATES) is None:
            n_fb += 1
    return n_fb, n_total

print('Fallback rate (невалидный JSON → исходный порядок):')
for name, key in [('ALS', 'als_raw'), ('NCF', 'ncf_raw'), ('LightFM', 'lightfm_raw')]:
    fb, tot = fallback_count(rerank_cache[key])
    print(f'  {name}: {fb}/{tot} ({fb/tot*100:.1f}%)')

In [ ]:
als_recs_base = {int(uid): [iid for iid, _ in als_candidates[int(uid)][:FINAL_K]] for uid in sample_users}
ncf_recs_base = {int(uid): [iid for iid, _ in ncf_candidates[int(uid)][:FINAL_K]] for uid in sample_users}
lfm_recs_base = {int(uid): [iid for iid, _ in lfm_candidates[int(uid)][:FINAL_K]] for uid in sample_users}
print('Базовые recs (top-FINAL_K) готовы для всех трёх источников')

In [ ]:
K_VALUES = [5, 10, 20]


def precision_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(rec_k) if rec_k else 0.0


def recall_at_k(recommended, relevant, k):
    rec_k = set(recommended[:k])
    return len(rec_k & set(relevant)) / len(set(relevant)) if relevant else 0.0


def map_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    score, hits = 0.0, 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            score += hits / (i + 1)
    return score / min(len(relevant), k)


def ndcg_at_k(recommended, relevant, k):
    relevant = set(relevant)
    if not relevant:
        return 0.0
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(recommended[:k]) if item in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0


def aggregate(recs_by_user, users):
    out = {k: {m: [] for m in ['precision', 'recall', 'map', 'ndcg']} for k in K_VALUES}
    for u in users:
        u = int(u)
        rel = test_user_items.get(u, [])
        if not rel:
            continue
        rec = recs_by_user.get(u, [])
        for k in K_VALUES:
            out[k]['precision'].append(precision_at_k(rec, rel, k))
            out[k]['recall'].append(recall_at_k(rec, rel, k))
            out[k]['map'].append(map_at_k(rec, rel, k))
            out[k]['ndcg'].append(ndcg_at_k(rec, rel, k))
    return {k: {m: float(np.mean(v)) if v else 0.0 for m, v in metrics.items()}
            for k, metrics in out.items()}

In [ ]:
all_systems = {
    'ALS':                  als_recs_base,
    'ALS + LLM rerank':     als_recs_rerank,
    'NCF':                  ncf_recs_base,
    'NCF + LLM rerank':     ncf_recs_rerank,
    'LightFM':              lfm_recs_base,
    'LightFM + LLM rerank': lfm_recs_rerank,
}

if ZERO_SHOT_CACHE.exists():
    try:
        with open(ZERO_SHOT_CACHE, 'r', encoding='utf-8') as f:
            zs = json.load(f)
        sample_set = set(int(u) for u in sample_users)
        zs_recs = {int(k): v for k, v in zs.get('recs', {}).items() if int(k) in sample_set}
        if zs_recs:
            all_systems['LLM zero-shot'] = zs_recs
            print(f'LLM zero-shot подключён: {len(zs_recs)} пересечений с sample_users')
    except Exception as e:
        print(f'Не удалось подгрузить zero-shot cache: {e}')

results_all = {name: aggregate(recs, [int(u) for u in sample_users])
               for name, recs in all_systems.items()}

In [ ]:
rows = []
for name, res in results_all.items():
    row = {'model': name}
    for k in K_VALUES:
        for m in ('precision', 'recall', 'map', 'ndcg'):
            row[f'{m}@{k}'] = res[k][m]
    rows.append(row)
comp_df = pd.DataFrame(rows).set_index('model')
print('Сводная таблица метрик:')
display(comp_df.style.format('{:.4f}'))

delta_rows = []
for base_name in ['ALS', 'NCF', 'LightFM']:
    rerank_name = f'{base_name} + LLM rerank'
    delta = {
        f'{m}@{k}': (results_all[rerank_name][k][m] - results_all[base_name][k][m])
                   / (results_all[base_name][k][m] + 1e-12) * 100
        for k in K_VALUES for m in ('precision', 'recall', 'map', 'ndcg')
    }
    delta_rows.append({'pair': f'{base_name} → +rerank', **delta})
delta_df = pd.DataFrame(delta_rows).set_index('pair')
print('\nПрирост от LLM rerank (% относительно базы):')
display(delta_df.style.format('{:+.1f}%'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = {
    'ALS': 'tab:blue', 'ALS + LLM rerank': 'tab:blue',
    'NCF': 'tab:orange', 'NCF + LLM rerank': 'tab:orange',
    'LightFM': 'tab:green', 'LightFM + LLM rerank': 'tab:green',
    'LLM zero-shot': 'tab:red',
}
for ax, metric in zip(axes.flat, ['precision', 'recall', 'map', 'ndcg']):
    for name, res in results_all.items():
        is_rerank = 'rerank' in name or name == 'LLM zero-shot'
        ax.plot(
            K_VALUES,
            [res[k][metric] for k in K_VALUES],
            marker='s' if is_rerank else 'o',
            linestyle='-' if is_rerank else '--',
            color=colors.get(name, 'gray'),
            label=name,
            linewidth=2,
        )
    ax.set_xlabel('K')
    ax.set_ylabel(f'{metric.upper()}@K')
    ax.set_title(f'{metric.upper()}@K')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
plt.suptitle(f'LLM re-ranking: base (--) vs base+rerank (—) (n={len(sample_users)})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
uid = int(sample_users[0])
print(f'=== uid={uid} ===\n')
print(f'История ({len(user_history_cache.get(uid, []))} покупок, последние {MAX_HISTORY_ITEMS}):')
print(build_user_history(uid)[:600])
print('\n--- ALS top-5 (base) ---')
for iid, sc in als_candidates[uid][:5]:
    print(f'  {iid}: {item_name_map.loc[iid][:80]}  (score={sc:.3f})')
print('\n--- LLM raw для ALS (первые 250 симв.) ---')
print(rerank_cache['als_raw'].get(uid, '')[:250])
print('\n--- ALS top-5 после rerank ---')
for iid in als_recs_rerank[uid][:5]:
    print(f'  {iid}: {item_name_map.loc[iid][:80]}')
print('\n--- Test items (первые 10) ---')
for iid in test_user_items.get(uid, [])[:10]:
    print(f'  {iid}: {item_name_map.loc[iid][:80]}')

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd().resolve()
for _cand in [_here, *_here.parents]:
    _utils = _cand / 'source' / 'utils'
    if _utils.exists():
        sys.path.insert(0, str(_utils))
        break
import importlib, timing as _t
importlib.reload(_t)
from timing import benchmark_user_latency, save_benchmark, detect_hardware, benchmark_amortized_total

rng = np.random.RandomState(42)
_users_arr = np.array(sample_users)
bench_size = min(2000, len(_users_arr))
bench_users = sorted(map(int, rng.choice(_users_arr, size=bench_size, replace=False).tolist()))
print(f'Benchmark на {len(bench_users)} users (warmup=50, k=10, K_CANDIDATES={K_CANDIDATES})')

import torch
_use_cuda = torch.cuda.is_available()
DATASET_META = {'n_users': int(n_users), 'n_items': int(n_items),
                'n_candidates': int(K_CANDIDATES), 'final_k': int(FINAL_K)}
NB_PATH = 'source/collaborative_filtering/llm_rerank_vllm.ipynb'

In [ ]:
import implicit
from implicit.als import AlternatingLeastSquares as ALS_CPU

_als_cpu = ALS_CPU(factors=50, regularization=0.01, iterations=15,
                   calculate_training_loss=False, random_state=RANDOM_SEED)
_als_cpu.fit(train_matrix, show_progress=False)

def als_cpu_stage1(uid):
    item_ids, scores = _als_cpu.recommend(
        uid, train_matrix[uid], N=K_CANDIDATES, filter_already_liked_items=True
    )
    return list(item_ids)

stats_als_cpu = benchmark_user_latency(
    als_cpu_stage1, bench_users, warmup=50, k=10, sync_cuda=False, label='LLM-rerank stage1 ALS-CPU'
)
print(f"ALS-CPU stage1: mean={stats_als_cpu['mean_ms']:.3f}ms p50={stats_als_cpu['p50_ms']:.3f} p95={stats_als_cpu['p95_ms']:.3f}")
save_benchmark(stats_als_cpu, model_name='ALS+LLM-rerank', stage='stage1',
    hardware=detect_hardware(prefer='cpu'), dataset_meta=DATASET_META,
    extra={'library': 'implicit', 'use_gpu': False, 'stage_desc': 'ALS top-50'},
    notebook=NB_PATH, n_items=n_items)

try:
    has_cuda_implicit = bool(implicit.gpu.HAS_CUDA)
except Exception:
    has_cuda_implicit = False
if has_cuda_implicit:
    from implicit.gpu.als import AlternatingLeastSquares as ALS_GPU
    factors_gpu = ((50 + 31) // 32) * 32
    _als_gpu = ALS_GPU(factors=factors_gpu, regularization=0.01, iterations=15,
                       random_state=RANDOM_SEED)
    _als_gpu.fit(train_matrix, show_progress=False)

    def als_gpu_stage1(uid):
        item_ids, scores = _als_gpu.recommend(
            uid, train_matrix[uid], N=K_CANDIDATES, filter_already_liked_items=True
        )
        return list(item_ids)

    stats_als_gpu = benchmark_user_latency(
        als_gpu_stage1, bench_users, warmup=50, k=10, sync_cuda=False, label='LLM-rerank stage1 ALS-GPU'
    )
    print(f"ALS-GPU stage1: mean={stats_als_gpu['mean_ms']:.3f}ms p50={stats_als_gpu['p50_ms']:.3f} p95={stats_als_gpu['p95_ms']:.3f}")
    save_benchmark(stats_als_gpu, model_name='ALS+LLM-rerank', stage='stage1',
        hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
        extra={'library': 'implicit', 'use_gpu': True, 'factors': factors_gpu,
               'stage_desc': 'ALS top-50'},
        notebook=NB_PATH, n_items=n_items)
else:
    print('Skipping ALS-GPU stage1 (implicit GPU not available)')

In [ ]:
ncf_model.eval()
ncf_model.to(device)
import torch
_items_t_full = torch.arange(n_items, device=device, dtype=torch.long)

@torch.no_grad()
def ncf_stage1(uid):
    u = torch.full_like(_items_t_full, fill_value=int(uid))
    scores = ncf_model(u, _items_t_full)
    bought = user_train_items_cache.get(int(uid), set())
    if bought:
        idx = [it for it in bought if it < n_items]
        if idx:
            scores[torch.tensor(idx, device=device, dtype=torch.long)] = float('-inf')
    top_idx = torch.topk(scores, K_CANDIDATES).indices.cpu().numpy()
    return list(top_idx)

stats_ncf = benchmark_user_latency(
    ncf_stage1, bench_users, warmup=100, k=10, sync_cuda=(device.type == 'cuda'),
    label='LLM-rerank stage1 NCF'
)
print(f"NCF stage1 ({device}): mean={stats_ncf['mean_ms']:.3f}ms p50={stats_ncf['p50_ms']:.3f} p95={stats_ncf['p95_ms']:.3f}")
save_benchmark(stats_ncf, model_name='NCF+LLM-rerank', stage='stage1',
    hardware=detect_hardware(prefer='gpu') if device.type == 'cuda' else detect_hardware(prefer='cpu'),
    dataset_meta=DATASET_META,
    extra={'library': 'torch_neumf', 'sync_cuda': device.type == 'cuda',
           'stage_desc': 'NCF top-50'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
_all_items_arr = np.arange(n_items)

def lfm_stage1(uid):
    scores = lfm.predict(int(uid), _all_items_arr, num_threads=1)
    scores = np.nan_to_num(scores, nan=-np.inf)
    bought = user_train_items_cache.get(int(uid), set())
    if bought:
        scores[list(bought)] = -np.inf
    top_idx = np.argpartition(-scores, K_CANDIDATES)[:K_CANDIDATES]
    top_idx = top_idx[np.argsort(-scores[top_idx])]
    return list(top_idx)

stats_lfm = benchmark_user_latency(
    lfm_stage1, bench_users, warmup=50, k=10, sync_cuda=False, label='LLM-rerank stage1 LightFM'
)
print(f"LightFM stage1: mean={stats_lfm['mean_ms']:.3f}ms p50={stats_lfm['p50_ms']:.3f} p95={stats_lfm['p95_ms']:.3f}")
save_benchmark(stats_lfm, model_name='LightFM+LLM-rerank', stage='stage1',
    hardware=detect_hardware(prefer='cpu'), dataset_meta=DATASET_META,
    extra={'library': 'lightfm', 'num_threads': 1, 'stage_desc': 'LightFM top-50'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
from vllm import SamplingParams
_sp = SamplingParams(temperature=0.0, max_tokens=400, stop_token_ids=LLM_TERMINATORS)

N_BATCH = min(500, len(bench_users))
batch_users_for_llm = bench_users[:N_BATCH]
batch_prompts = [
    build_rerank_prompt(uid, [iid for iid, _ in als_candidates[uid][:K_CANDIDATES]])
    for uid in batch_users_for_llm
]

_ = llm.generate(batch_prompts[:2], _sp)

import time
t0 = time.perf_counter()
_ = llm.generate(batch_prompts, _sp)
elapsed_batch = time.perf_counter() - t0
stats_llm_batch = benchmark_amortized_total(
    elapsed_batch, len(batch_prompts),
    label='LLM-rerank batch_amortized', k=10, batch_size=len(batch_prompts),
    note='vLLM continuous batching; per-prompt timing not exposed'
)
print(f"LLM stage2 batch ({len(batch_prompts)} prompts): {elapsed_batch:.1f}s, mean={stats_llm_batch['mean_ms']:.1f}ms/prompt")
save_benchmark(stats_llm_batch, model_name='LLM-rerank-stage', stage='llm_rerank',
    hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
    extra={'library': 'vllm', 'mode': 'batch_amortized',
           'continuous_batching': True, 'sync_cuda': True,
           'stage_desc': 'vLLM rerank K=50 candidates, batched'},
    notebook=NB_PATH, n_items=n_items)

In [ ]:

N_SINGLE = 30
single_users_for_llm = bench_users[:N_SINGLE]
single_prompts = [
    build_rerank_prompt(uid, [iid for iid, _ in als_candidates[uid][:K_CANDIDATES]])
    for uid in single_users_for_llm
]

_ = llm.generate(single_prompts[:1], _sp)

import time
latencies_ms = np.empty(len(single_prompts), dtype=np.float64)
for i, p in enumerate(single_prompts):
    t0 = time.perf_counter()
    _ = llm.generate([p], _sp)
    latencies_ms[i] = (time.perf_counter() - t0) * 1000.0

import numpy as _np
stats_llm_single = {
    'mean_ms': float(latencies_ms.mean()),
    'p50_ms': float(_np.percentile(latencies_ms, 50)),
    'p95_ms': float(_np.percentile(latencies_ms, 95)),
    'total_s': float(latencies_ms.sum() / 1000.0),
    'throughput_ups': float(len(latencies_ms) * 1000.0 / latencies_ms.sum()) if latencies_ms.sum() > 0 else 0.0,
    'label': 'LLM-rerank single', 'k': 10, 'n_users': len(latencies_ms),
    'warmup': 1, 'latencies_ms': latencies_ms.tolist(), 'mode': 'single_user',
    'note': 'batch=1 — pessimistic (no continuous batching)',
}
print(f"LLM stage2 single: mean={stats_llm_single['mean_ms']:.1f}ms p50={stats_llm_single['p50_ms']:.1f} p95={stats_llm_single['p95_ms']:.1f}")
save_benchmark(stats_llm_single, model_name='LLM-rerank-stage', stage='llm_rerank',
    hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
    extra={'library': 'vllm', 'mode': 'single_user',
           'continuous_batching': False, 'sync_cuda': True},
    notebook=NB_PATH, n_items=n_items)

In [ ]:
llm_amortized_ms = stats_llm_batch['mean_ms']

for base_name, stats_s1, model_name in [
    ('ALS-CPU', stats_als_cpu, 'ALS+LLM-rerank'),
    ('NCF', stats_ncf, 'NCF+LLM-rerank'),
    ('LightFM', stats_lfm, 'LightFM+LLM-rerank'),
]:
    e2e_lat = np.array(stats_s1['latencies_ms']) + llm_amortized_ms
    stats_e2e = {
        'mean_ms': float(e2e_lat.mean()),
        'p50_ms': float(np.percentile(e2e_lat, 50)),
        'p95_ms': float(np.percentile(e2e_lat, 95)),
        'total_s': float(e2e_lat.sum() / 1000.0),
        'throughput_ups': float(len(e2e_lat) * 1000.0 / e2e_lat.sum()),
        'label': f'{base_name}+LLM e2e', 'k': 10,
        'n_users': len(e2e_lat), 'warmup': stats_s1['warmup'],
        'latencies_ms': e2e_lat.tolist(), 'mode': 'single_user',
        'note': 'stage1 measured + LLM amortized per-prompt',
    }
    print(f"{model_name} e2e: mean={stats_e2e['mean_ms']:.1f}ms p50={stats_e2e['p50_ms']:.1f} p95={stats_e2e['p95_ms']:.1f}")
    save_benchmark(stats_e2e, model_name=model_name, stage='e2e',
        hardware=detect_hardware(prefer='gpu'), dataset_meta=DATASET_META,
        extra={'base': base_name, 'composition': 'stage1 + llm_amortized',
               'llm_amortized_ms': llm_amortized_ms},
        notebook=NB_PATH, n_items=n_items)